# Validation du modèle d'inter/extrapolation

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import rioxarray
import xarray as xr

### Fonctions utilisées

**mae_date** renvoie la MAE entre deux xr.DataArray, 
Dans notre cas, les xr.DataArray correspondent à la distribution spatiale de l'évapo-transpiration réelle ou simulée pour une date. Dans le calcul de la MAE, les individus comparés sont donc les valeurs de l'évapo-transpiration réelle $x_i$ et simulée $y_i$ au pixel $i$.

In [ ]:
def mae_date(x: xr.DataArray, y: xr.DataArray) -> float:
    n = x.shape[0] * x.shape[1]
    mae = (abs(x - y) / n).sum()
    return mae.item()

De la même façon, **me** et **rmse_date** calculent respectivement l'erreur moyenne et la RMSE entre deux xr.DataArray.

In [ ]:
def rmse_date(x: xr.DataArray, y: xr.DataArray) -> float:
    n = x.shape[0] * x.shape[1]
    se = ((x - y) ** 2).sum()
    rmse = np.sqrt(se / n)
    return rmse.item()

In [ ]:
def me_date(x: xr.DataArray, y: xr.DataArray) -> float:
    n = x.shape[0] * x.shape[1]
    mae = ((x - y) / n).sum()
    return mae.item()

**relative_error_date** calcule l'erreur relative entre deux xr.DataArray. En pratique, cette mesure n'est pas pertinente puisque sa valeur explose lorsque les pixels possèdent une très petite evapo-transpiration.

In [ ]:
def relative_error_date(x: xr.DataArray, y: xr.DataArray) -> xr.DataArray:
    return (x - y) / y

**get_et_single_date** retourne l'évapo-transpiration réelle pour une date (xr.DataArray). Le fichier .tif correspondant doit être stocké dans le répertoire **dir**.

In [ ]:
def get_et_single_date(dir: str, date: dt.datetime) -> xr.DataArray:  # noqa: A002
    filename = f"et_single_date_{date.strftime('%Y%m%d')}.tif"
    path = os.path.join(dir, filename)
    data = rioxarray.open_rasterio(path)
    return data[0, :, :]

**get_et_time_series** retourne l'évapo-transpiration simulée pour une date (xr.DataArray).

In [ ]:
def get_et_time_series(dir: str, date: dt.datetime) -> xr.DataArray:  # noqa: A002
    filename = f"et_time_series_{date.strftime('%Y%m%d')}.tif"
    path = os.path.join(dir, filename)
    data = rioxarray.open_rasterio(path)
    return data[0, :, :]

**similitude_image** renvoie la liste des valeurs pour une mesure d'erreur **metric** donnée (mae_date, rmse_date ou relative_error_date) entre l'évapo-transpiration réelle et simulée sur une période de temps donnée (**t_start** - **t_end**). Les fichiers .tif associés doivent être stockés dans le répertoire **dir**.

In [ ]:
def error_list(dir: str, t_start: str, t_end: str, metric) -> list[float]:  # noqa: A002
    rmse_list = []
    time = xr.date_range(t_start, freq="1D", end=t_end)
    for date in time:
        rmse_date = metric(
            get_et_single_date(dir, date), get_et_time_series(dir, date)
        )
        rmse_list.append(rmse_date)
    return rmse_list

**mae_pixel** renvoie la distribution spatiale (xr.DataArray) de la MAE calculée pour chaque pixel sur une période temps donnée. Dans le calcul de la MAE d'un pixel, les individus comparés sont donc les valeurs en ce pixel de l'évapo-transpiration réelle $x_i$ et simulée $y_i$ à la date $i$.

In [ ]:
def mae_pixel(dir: str, t_start: str, t_end: str) -> xr.DataArray:  # noqa: A002
    time = xr.date_range(t_start, freq="1D", end=t_end)
    mae_map = sum(
        abs(get_et_single_date(dir, date) - get_et_time_series(dir, date))
        for date in time
    )
    return mae_map / len(time)

**me_pixel** renvoie la distribution spatiale de l'erreur moyenne ME pour chaque pixel. On peut alors exploiter le signe de l'erreur moyenne.

In [ ]:
def me_pixel(dir: str, t_start: str, t_end: str):  # noqa: A002
    time = xr.date_range(t_start, freq="1D", end=t_end)
    mae_map = sum(
        get_et_single_date(dir, date) - get_et_time_series(dir, date)
        for date in time
    )
    return mae_map / len(time)

### Données

In [ ]:
dir_savanes_3 = "./data_validation/data_savanes"
dir_senegal = "./data_validation/data_senegal"
dir_sud = "./data_validation/data_sud"
dir_sud_ete = "./data_validation/data_sud_ete"
dir_stg = "./data_validation/data_stgirons"
dir_savanes_2 = "./data_validation/data_savanes2"
dir_savanes_4 = "./data_validation/data_savanes3"

### Analyse spatiale de l'erreur

On commence par faire des tests sur la région du Sénégal au mois de mars. La région présente une grande variété de types végétations et sols, qui pourraient potentiellement influer sur la distribution spatiale de l'erreur. Aussi, sur cette période, aucun aléa liée à la pluie ne viendra perturbé le modèle d'interpolation : le ciel est toujours dégagé et le rayonnement solaire fort.

In [ ]:
img = mpimg.imread("./data_validation/map.png")
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
me_pxl_senegal = me_pixel(dir_senegal, "2023-03-01", "2023-03-10")
mae_pxl_senegal = mae_pixel(dir_senegal, "2023-03-01", "2023-03-10")

In [ ]:
get_et_single_date(
    dir_senegal,
    dt.datetime.strptime("2023-03-01", "%Y-%m-%d"),  # noqa: DTZ007
).plot(vmax=1)
plt.title("Evapo-transpiration réelle le 01/03/2023")

In [ ]:
mae_pxl_senegal.plot()
plt.title("MAE Sénégal")

In [ ]:
me_pxl_senegal.plot(cmap="seismic", vmin=-0.1, vmax=0.1)
plt.title("ME Sénégal - bornée entre -0.1 et 0.1")

In [ ]:
mae_pxl_senegal.mean().item()

La MAE par pixel moyenne est très petite (~2.4%). \
L'erreur est concentrée sur les zones d'eau : on distingue le tracé des fleuve et les bassins d'eaux libres. Cela s’explique sans doute par un problème de trop faible résolution du masque d’eau pour les données d’entrée : l'interpolation n’a pas de sens lorsqu’elle est appliqué aux surfaces aquatiques.

Pour étudier les variations de l'erreur en limitant l'effet de ces valeurs extrêmes, on borne la ME entre -0.1 et 0.1. On remarque une erreur croissante au voisinage des zones d'eau. Puisqu'on interpole chaque pixel indivduellement sans tenir compte de son contexte spatial, cela signifie que, au delà du problème lié à la résolution du masque d'eau, la présence de zones d'eau pertube le modèle d'un point de vue physique.

En étudiant le signe de l'erreur, il semble que l'évapo-transpiration soit (légérement) sur-estimée le long des côtes. Nous n'observons pas de tendance commune au voisinage des zones d'eaux : l'évapo-transpiration est tantôt sur-estimée (bassins du nord, fleuve du sud-est), tantôt sous-estimées (fleuves côtiers, bassins du sud-est). \
Concernant le type de végétations, il semble que l'évapo-transpiration soit-estimée dans les zones de végétations denses (savanes boisées et fôrets du sud) et sous-estimée dans les zones de cultures. Les erreurs associées restent minimes toutefois (> 2.5 % en valeur absolue)

Pour continuer notre analyse de l'erreur d'un point de vue spatial, nous étudions une zone aux conditions climatiques et géographiques moins idéales : la région PACA. En plus de la présence de hautes montagnes, on compte sur la période quelques jours couverts et pluvieux.

In [ ]:
mae_pxl_sud = mae_pixel(dir_sud, "2023-03-01", "2023-03-09")
me_pxl_sud = me_pixel(dir_sud, "2023-03-01", "2023-03-09")

In [ ]:
get_et_single_date(
    dir_sud,
    dt.datetime.strptime("2023-03-02", "%Y-%m-%d"),  # noqa: DTZ007
).plot()
plt.title("Evapo-transpiration réelle le 02/03/2023")

In [ ]:
mae_pxl_sud.plot()
plt.title("MAE PACA")

In [ ]:
me_pxl_sud.plot(cmap="seismic", vmin=-0.5, vmax=0.5)
plt.title("ME PACA - bornée - bornée entre -0.5 et 0.5")

In [ ]:
mae_pxl_sud.mean().item()

La MAE par pixel moyenne est plus grande qu'au Sénégal (~12.6%). \
Les zones d'eau, plus petites, ne semblent plus avoir d'effet significatifs. On distingue tout de même les lacs en bas des Alpes lorsqu'on borne la ME (< 2% d'erreur en valeur absolue). \
Hormis les Alpes qui se distinguent nettement, nous n'avons pas réussi à identifier de facteurs liés aux types de végétations et de sols ou aux conditions métérologique qui expliqueraient la distribution spatiale de l'erreur.\
En ce qui concerne les Alpes, on observe que l'erreur presque nulle s'accompagne d'une évapo-transpiration elle aussi presque nulle. Ce résultat peut s'expliquer par l'absence de phénomène d'évaporation significatifs dans cette région en hiver, absence que le modèle d'interpolation parvient à bien capturer grâce à un flux de chaleur latente LE très faible.

Intéressons nous maintenant à ce qui se passe l'été.

In [ ]:
mae_pxl_sud_ete = mae_pixel(dir_sud_ete, "2023-07-01", "2023-07-09")
me_pxl_sud_ete = me_pixel(dir_sud_ete, "2023-07-01", "2023-07-09")

In [ ]:
get_et_single_date(
    dir_sud_ete,
    dt.datetime.strptime("2023-07-05", "%Y-%m-%d"),  # noqa: DTZ007
).plot()
plt.title("Evapo-transpiration réelle le 05/07/2023")

In [ ]:
mae_pxl_sud_ete.plot(vmax=0.8)
plt.title("MAE PACA été")

In [ ]:
me_pxl_sud_ete.plot(cmap="seismic", vmin=-0.5, vmax=0.5)
plt.title("ME PACA été - bornée - bornée entre -0.5 et 0.5")

Lorsqu'on affiche l'erreur pour le début du mois de juillet, on note que celle-ci est particulièrement forte dans les Alpes (évapo-transpiration fortement surestimée), alors qu'elle était quasiment nulle dans cette région en mars. Le modèle d'interpolation n'est pas adapté à la dynamique estivale des hautes montagnes. Ceci pourrait s'expliquer du fait de la fonte des neiges ou des glaces qui a lieu à cette période.\
La relation entre rayonnement solaire et evapo-transpiration utilisée lors de l'interpolation dérive d'un bilan energétique faisant intervenir principalement le flux de chaleur latente LE. Ce flux correspond à l'énergie absorbée lors de la fusion ou évaporation de l'eau. Notre modèle estime que LE quantifie énegétiquement l'évapo-transpiration. Or, en été, une part signicative du LE sert à la fusion des neiges et glaces plutôt qu'à l'évaporation de l'eau liquide. Cette répartition imprévue de l'énérgie  conduit alors le modèle à surestimer l'évapo-transpiration simulée.

### Analyse temporelle de l'erreur


Dans un premier temps, nous souhaitons étudier les effets de la fréquence d'acquisition de l'évapo-transpiration "réelle" sur l'erreur.
Nous choisissons pour cela une ROI centrée sur les savanes boisées du Sénégal au mois de mars 2023. Le type de végétation y est très homogène et les conditions climatiques nous permettent d'éviter les aléas liés à la pluie.


In [ ]:
time = xr.date_range("2023-03-01", end="2023-03-30", freq="1D")

In [ ]:
mae_savanes_4 = error_list(dir_savanes_4, "2023-03-01", "2023-03-30", mae_date)
plt.bar(time, mae_savanes_4)
plt.xticks(time[::6])
plt.title("MAE journalière du 01 au 30/03/2023 - Acquisition tous les 4 jours")

In [ ]:
max_mae4 = max(mae_savanes_4)
max_mae4

In [ ]:
mean_mae_4 = sum(mae_savanes_4) / len(time)
mean_mae_4

In [ ]:
mae_savanes_3 = error_list(dir_savanes_3, "2023-03-01", "2023-03-30", mae_date)
plt.bar(time, mae_savanes_3)
plt.xticks(time[::6])
plt.title("MAE journalière du 01 au 30/03/2023 - Acquisition tous les 3 jours")

In [ ]:
max_mae3 = max(mae_savanes_3)
max_mae3

In [ ]:
mean_mae_3 = sum(mae_savanes_3) / len(time)
mean_mae_3

In [ ]:
mae_savanes_2 = error_list(dir_savanes_2, "2023-03-01", "2023-03-30", mae_date)
plt.bar(time, mae_savanes_2)
plt.xticks(time[::6])
plt.title("MAE journalière du 01 au 30/03/2023 - Acquisition tous les 2 jours")

In [ ]:
max_mae2 = max(mae_savanes_2)
max_mae2

In [ ]:
mean_mae_2 = sum(mae_savanes_2) / len(time)
mean_mae_2

On remarque que, comme attendu, plus les acquisitions sont espacées, plus l'erreur moyenne est grande (2.3% pour 4 jours, 1.6% pour 3 jours et 1.1% pour 2 jours). \
Néanmoins, les valeurs maximales de l'erreur semblent peu impactées : pour 4 jours, le pic d'erreur n'est que de 7.6% alors qu'il est de 7.8% pour 3 jours.

Nous souhaitons maintenant étudier les effets de la pluie, variable qui n'est pas prise directement en compte dans le modèle d'interpolation, sur l'erreur. Pour cela, nous choisissons une ROI centrée sur les environs de Saint-Girons (Ariège). Nous lançons l'algorithme d'interpolation avec acquisition tous les 4 jours.

In [ ]:
time = xr.date_range("2023-03-01", end="2023-05-05", freq="1D")

Nous avons téléchargé la précipitation totale locale (ERA5-Land) pour pouvoir la comparer à la MAE en fonction des dates.

In [ ]:
data_precip = xr.open_dataset("./data_validation/pluie.nc")
# Precipiation par heure à sommer (produit non accumulé) pour obtenir celle de la journée, conversion en mm
data_precip = (
    data_precip.groupby(data_precip["valid_time"].dt.date).sum() * 1000
)

In [ ]:
rmse_stg = error_list(dir_stg, "2023-03-01", "2023-05-05", rmse_date)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(time, rmse_stg, color="tab:orange")
axes[0].set_title("RMSE Saint-Girons")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("RMSE")
axes[0].set_xticks(time[::15])
axes[1].bar(time, data_precip.tp, color="tab:blue")
axes[1].set_title("Precipitation")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Total precipitation (mm)")
axes[1].set_xticks(time[::15])
plt.tight_layout()
plt.show()

D'après les barplots ci-dissus, il semble que la RMSE et les précipitations journalières évoluent de façon similaire : plus il a plu dernièrement et plus l'interpolation serait mauvaise. Pour confirmer cette observation, nous calculons la corrélation entre la RMSE (pour les dates ou l'évapo-transpiration est simulée) et la précipitation journalière entre l'avant-veille et quatre jours avant.

In [ ]:
# On sélectionne uniquement les MAE correspondants aux interpolations et extrapolations
rmse_simul = []
for i in range(2, len(rmse_stg)):
    if i % 4 != 0:
        rmse_simul.append(rmse_stg[i])  # noqa: PERF401

In [ ]:
# On sélectionne pour chaque date la somme des précipitations des 4 jours précédents
precip_simul = []
for i in range(2, data_precip.tp.shape[0]):
    if i % 4 != 0:
        if i == 2:
            precip_simul.append(data_precip.tp[0].item())
        elif i == 3:
            precip_simul.append(
                data_precip.tp[0].item() + data_precip.tp[1].item()
            )
        else:
            precip_simul.append(
                data_precip.tp[i - 4].item()
                + data_precip.tp[i - 3].item()
                + data_precip.tp[i - 2].item()
            )

In [ ]:
np.corrcoef(rmse_simul, precip_simul)[0, 1].item()

La corrélation entre les deux variables est assez forte (0.70), ce qui nous laisse pense qu'il existe bien un lien entre elles. Pour en déduire une éventuelle interprétation physique, nous privilégions par la suite l'étude de la ME plutôt que de la RMSE, qui ne fournissait pas d'informations sur le signe de l'erreur.

In [ ]:
me_stg = error_list(dir_stg, "2023-03-01", "2023-05-05", me_date)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(time, me_stg, color="tab:orange")
axes[0].set_title("ME Saint-Girons")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("ME")
axes[0].set_xticks(time[::15])
axes[1].bar(time, data_precip.tp, color="tab:blue")
axes[1].set_title("Precipitation")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Total precipitation (mm)")
axes[1].set_xticks(time[::15])
plt.tight_layout()
plt.show()

In [ ]:
# On sélectionne uniquement les MAE correspondants aux interpolations et extrapolations
me_simul = []
for i in range(2, len(me_stg)):
    if i % 4 != 0:
        me_simul.append(me_stg[i])  # noqa: PERF401

In [ ]:
# On sélectionne pour chaque date la somme des précipitations des 4 jours précédents
precip_simul = []
for i in range(2, data_precip.tp.shape[0]):
    if i % 4 != 0:
        if i == 2:
            precip_simul.append(data_precip.tp[0].item())
        elif i == 3:
            precip_simul.append(
                data_precip.tp[0].item() + data_precip.tp[1].item()
            )
        else:
            precip_simul.append(
                data_precip.tp[i - 4].item()
                + data_precip.tp[i - 3].item()
                + data_precip.tp[i - 2].item()
            )

In [ ]:
corr = np.corrcoef(me_simul, precip_simul)[0, 1]

In [ ]:
corr.item()

Le lien entre erreur et précipations les jours précédents semble un peu moins évident sur les barplots en utilisant la ME. Néanmoins, on trouve que la ME pour un jour J et les précipations entre J-2 et J-4 sont assez fortement corrélées négativement (-0.66). Ainsi, de fortes précipations les jours précédents l'interpolation favorisent la surestimation de l'évapo-transpiration. \
Nous pouvons supposer que, comme dans le cas de la fonte des glaces ou des neiges, cette surestimation de l'évapo-transpiration est liée à une mauvaise interprétation du flux de chaleur latente LE par le modèle. Une part ignificative de ce flux sert à l'évaporation de l'eau contenu dans le sol, particulièrment humide à cause des dernières précipiations, plutôt qu' à la simple évapo-transpiration de la végétation.

In [ ]:
sum(me_stg[0:29]) / 30

### Conclusion

Les résultats de l'algorithme d'interpolation sont particulièrement satisfaisants pour les conditions géographiques et climatiques idéales du Sénégal. Les seules erreurs significatives concernent le voisinage des zones d'eau, en raison à la fois de la résolution insuffisante du masque d'eau des données en entrée et de la dynamique physique complexe de l'eau propre à ces milieux (pas de tendance claire en terme de surestimation ou sousestimation).
Aussi, nous avons pu voir que l'allongement de la durée entre deux acquisitions consécutives n'entraine qu'une augmenation minime de l'erreur moyenne.

L'interpolation se dégrade pour des régions qui présentent des conditions moins favorables : pluie, présence de hautes montagnes. Nous avons supposé que l'erreur est principalement due à une interprétation excessivement simplifiée du flux de chaleur latente LE comme quantificateur énergétique de l'évapo-transpiration. Cette simplification ne tient en effet pas compte de dynamiques complexes du cycle de l'eau tels que l'humdification des sols après précipiations ou la fonte des glaces et des neiges.